# Australia Weather Dataset: Missing-Data Analysis & Preprocessing Pipeline

Goal: turn the raw *Rain in Australia* data (`weatherAUS.csv`, ~145k daily observations from 49 weather stations)
into a clean, fully numeric table that is ready for predicting **`RainTomorrow`**.

Pipeline:
1. Load and profile the data
2. Diagnose **where** and **when** values are missing
3. Remove stations whose records are too incomplete to trust
4. Impute numeric features (KNN) and categorical features (per-station forward/back fill)
5. Encode categoricals, validate, and export

Download the data from Kaggle ("Rain in Australia") and place `weatherAUS.csv` next to this notebook,
or set the `WEATHER_CSV` environment variable. In Google Colab the notebook mounts Drive automatically.

## 1. Setup

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ---------------- configuration ----------------
DATA_PATH = Path(os.environ.get("WEATHER_CSV", "weatherAUS.csv"))
FIG_DIR = Path("images")        # figures are saved here (used by the README)
RESULTS_DIR = Path("results")   # metrics are saved here
OUT_DIR = Path("data")          # cleaned dataset is saved here
SAVE_FIGS = True
SHOW_ALL_FEATURE_PLOTS = False  # True = one plot per feature (~40 extra figures)
MISSING_THRESHOLD = 0.80        # drop stations with any column missing more than this share
KNN_NEIGHBORS = 5
RANDOM_STATE = 42

# Optional: mount Google Drive when running in Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    # change this folder to wherever you stored weatherAUS.csv
    DATA_PATH = Path("/content/drive/MyDrive/Colab Notebooks/weatherAUS.csv")
except ImportError:
    pass

for d in (FIG_DIR, RESULTS_DIR, OUT_DIR):
    d.mkdir(exist_ok=True)

summary = {}  # collected metrics, saved to results/preprocessing_summary.json at the end

def save_fig(name):
    if SAVE_FIGS:
        plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

## 2. Load and profile the data

In [ ]:
weather_df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
# Sorting by station then date matters later (time-aware filling, time plots)
weather_df = weather_df.sort_values(["Location", "Date"]).reset_index(drop=True)

print(f"Dataset dimensions: {weather_df.shape}")
print(f"Stations: {weather_df['Location'].nunique()}")
print(f"Date range: {weather_df['Date'].min().date()} to {weather_df['Date'].max().date()}")
print()
print(weather_df.dtypes)

summary["raw_rows"], summary["raw_columns"] = weather_df.shape
summary["raw_locations"] = int(weather_df["Location"].nunique())

## 3. Missing-value overview

The percentage is computed directly from the data (the original notebook hard-coded the row count `145460`).

In [ ]:
nan_pct = (weather_df.isna().mean() * 100).sort_values(ascending=False)
print(nan_pct.round(2).to_string())
summary["nan_pct_by_column_raw"] = nan_pct.round(3).to_dict()

nan_nonzero = nan_pct[nan_pct > 0]
plt.figure(figsize=(12, 5))
plt.bar(nan_nonzero.index, nan_nonzero.values)
plt.xticks(rotation=60, ha="right")
plt.ylabel("Missing values (%)")
plt.title("Share of missing values per column (raw data)")
plt.tight_layout()
save_fig("00_missing_by_column_raw")
plt.show()

## 4. Where is data missing? (by station)

Missingness is not spread evenly. It is concentrated in a few columns (`Evaporation`, `Sunshine`, `Cloud*`, `Pressure*`)
and, within them, in particular stations.

In [ ]:
def nan_pct_by_location(df, feature):
    """Percent of missing values of `feature` for every Location (no deprecated groupby.apply)."""
    return df[feature].isna().groupby(df["Location"]).mean().mul(100).sort_values(ascending=False)

def plot_nan_by_location(df, feature, suffix="", fname=None):
    s = nan_pct_by_location(df, feature)
    plt.figure(figsize=(14, 6))
    plt.bar(s.index, s.values)
    plt.xticks(rotation=90)
    plt.ylabel("Percentage of NaN values (%)")
    plt.title(f"NaN Percentage in {feature} Column by Location{suffix}")
    plt.tight_layout()
    if fname:
        save_fig(fname)
    plt.show()

plot_nan_by_location(weather_df, "Evaporation", fname="01_evaporation_nan_by_location_raw")
plot_nan_by_location(weather_df, "Sunshine", fname="02_sunshine_nan_by_location_raw")

if SHOW_ALL_FEATURE_PLOTS:
    for feature in [c for c in weather_df.columns if c not in ("Location", "Date")]:
        plot_nan_by_location(weather_df, feature)

In [ ]:
# One picture for every feature x station: percent missing heatmap
def plot_nan_heatmap(df, title, fname):
    m = df.drop(columns=["Date"]).drop(columns=["Location"]).isna().groupby(df["Location"]).mean() * 100
    m = m.loc[m.mean(axis=1).sort_values(ascending=False).index]
    m = m.loc[:, m.max() > 0]
    fig, ax = plt.subplots(figsize=(13, max(5, 0.28 * len(m))))
    im = ax.imshow(m.values, aspect="auto", cmap="viridis", vmin=0, vmax=100)
    ax.set_xticks(range(m.shape[1])); ax.set_xticklabels(m.columns, rotation=60, ha="right")
    ax.set_yticks(range(m.shape[0])); ax.set_yticklabels(m.index)
    fig.colorbar(im, ax=ax, label="% missing")
    ax.set_title(title)
    plt.tight_layout()
    save_fig(fname)
    plt.show()

plot_nan_heatmap(weather_df, "Missing values (%) by station and feature, raw data", "03_missing_heatmap_raw")

## 5. When is data missing? (over time)

If values were missing at random, the indicator would flicker evenly. Instead, whole stretches vanish:
stations stop reporting an instrument. That matters for the imputation strategy.

*Fix:* the original code filtered Canberra/Williamtown but titled the second plot "Melbourne",
and plotted the series as sloped lines. This version takes the station as a parameter and uses step plots.

In [ ]:
def plot_nan_over_time(df, location, features, stage=""):
    sub = df[df["Location"] == location]
    for feature in features:
        plt.figure(figsize=(14, 3.5))
        plt.plot(sub["Date"], sub[feature].isna().astype(int), linewidth=0.8, drawstyle="steps-post")
        plt.title(f"NaN Pattern Over Time for '{feature}' in {location}{stage}")
        plt.xlabel("Date")
        plt.ylabel("NaN indicator (1 = missing)")
        plt.yticks([0, 1])
        plt.tight_layout()
        tag = "_" + stage.strip(" ()").lower().replace(" ", "_") if stage else ""
        save_fig(f"nan_over_time_{location}_{feature}{tag}")
        plt.show()

KEY_FEATURES = ["Evaporation", "Sunshine", "Cloud3pm"]
all_features = [c for c in weather_df.columns if c not in ("Location", "Date")]
plot_nan_over_time(weather_df, "Canberra", all_features if SHOW_ALL_FEATURE_PLOTS else KEY_FEATURES)

## 6. Remove stations with unusable records

A station is dropped if **any** column is missing more than 80% of the time. For those stations a model would learn
from imputed values rather than measurements.

In [ ]:
nan_frac_by_loc = weather_df.drop(columns="Location").isna().groupby(weather_df["Location"]).mean()
bad_locations = nan_frac_by_loc.index[(nan_frac_by_loc > MISSING_THRESHOLD).any(axis=1)].tolist()

weather_df_clean = weather_df[~weather_df["Location"].isin(bad_locations)].copy()

print("Locations before:", weather_df["Location"].nunique())
print("Locations removed:", bad_locations)
print("Locations remaining:", weather_df_clean["Location"].nunique())
print(f"Rows: {len(weather_df):,} -> {len(weather_df_clean):,} "
      f"({100 * len(weather_df_clean) / len(weather_df):.1f}% kept)")

summary.update(
    locations_removed=bad_locations,
    locations_remaining=int(weather_df_clean["Location"].nunique()),
    rows_after_location_removal=int(len(weather_df_clean)),
)

In [ ]:
plot_nan_by_location(weather_df_clean, "Evaporation", " (Post Removal)", "04_evaporation_nan_by_location_clean")
plot_nan_by_location(weather_df_clean, "Sunshine", " (Post Removal)", "05_sunshine_nan_by_location_clean")
plot_nan_heatmap(weather_df_clean, "Missing values (%) by station and feature, after station removal", "06_missing_heatmap_clean")

if SHOW_ALL_FEATURE_PLOTS:
    for feature in [c for c in weather_df_clean.columns if c not in ("Location", "Date")]:
        plot_nan_by_location(weather_df_clean, feature, " (Post Removal)")

plot_nan_over_time(weather_df_clean, "Williamtown",
                   all_features if SHOW_ALL_FEATURE_PLOTS else KEY_FEATURES, stage=" (Post Removal)")

## 7. Handle the target

`RainTomorrow` is what we want to predict, so it should never be *invented*. The original notebook forward-filled it
from neighbouring rows. Here rows without a label are dropped instead.

In [ ]:
n_before = len(weather_df_clean)
weather_df_clean = weather_df_clean.dropna(subset=["RainTomorrow"]).reset_index(drop=True)
print(f"Dropped {n_before - len(weather_df_clean):,} rows with no RainTomorrow label; {len(weather_df_clean):,} remain")
summary["rows_dropped_missing_target"] = int(n_before - len(weather_df_clean))

## 8. Split numeric and categorical features

In [ ]:
wind_cols = ["WindGustDir", "WindDir9am", "WindDir3pm"]
binary_cols = ["RainToday", "RainTomorrow"]
categorical_cols = ["Location"] + wind_cols + binary_cols

numeric_cols = [c for c in weather_df_clean.columns if c not in categorical_cols + ["Date"]]
numeric_df = weather_df_clean[numeric_cols]

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
assert len(numeric_cols) == 16, "unexpected numeric column set"

## 9. Numeric imputation with KNN

Features are standardized first (KNN uses distances, so unscaled `Pressure` would dominate), then each missing value is
filled from the 5 most similar days (`nan_euclidean`, distance-weighted), and the result is converted back to original units.
After imputation, values are clipped to the observed min/max and `Cloud*` (oktas) is rounded to whole numbers.

> **Leakage warning for modeling:** here the scaler and imputer see the whole dataset. When you train a model, fit them on the
> training split only (e.g. inside an sklearn `Pipeline`), otherwise test information leaks into training.

In [ ]:
scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(numeric_df)  # NaNs are ignored when fitting and preserved

imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS, weights="distance", metric="nan_euclidean")
numeric_imputed = imputer.fit_transform(numeric_scaled)
assert numeric_imputed.shape == numeric_scaled.shape, "KNNImputer dropped an all-missing column"

numeric_filled = pd.DataFrame(scaler.inverse_transform(numeric_imputed),
                              columns=numeric_cols, index=numeric_df.index)
numeric_filled = numeric_filled.clip(lower=numeric_df.min(), upper=numeric_df.max(), axis=1)
numeric_filled[["Cloud9am", "Cloud3pm"]] = numeric_filled[["Cloud9am", "Cloud3pm"]].round()

remaining = int(numeric_filled.isna().sum().sum())
print("Number of missing values after KNN imputation:", remaining)
summary["numeric_missing_after_imputation"] = remaining

In [ ]:
# Sanity check: did imputation distort the distributions?
check = pd.DataFrame({
    "missing_%": (numeric_df.isna().mean() * 100).round(2),
    "mean_observed": numeric_df.mean().round(3),
    "mean_after": numeric_filled.mean().round(3),
    "std_observed": numeric_df.std().round(3),
    "std_after": numeric_filled.std().round(3),
})
check["mean_shift_%"] = (100 * (check["mean_after"] - check["mean_observed"]) / check["mean_observed"].abs()).round(2)
display(check)
check.to_csv(RESULTS_DIR / "imputation_distribution_check.csv")

In [ ]:
# Hold-out validation: hide 5% of KNOWN values, impute them, and compare with the true values.
# Run on a 20k-row sample so it finishes quickly. Baseline = fill with the column mean.
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(numeric_scaled), size=min(20_000, len(numeric_scaled)), replace=False)
sample = numeric_scaled[idx].copy()

known = ~np.isnan(sample)
hide = known & (rng.random(sample.shape) < 0.05)
masked = sample.copy()
masked[hide] = np.nan

knn_filled = KNNImputer(n_neighbors=KNN_NEIGHBORS, weights="distance",
                        metric="nan_euclidean").fit_transform(masked)
mean_filled = np.where(np.isnan(masked), np.nanmean(masked, axis=0), masked)

rows = []
for j, col in enumerate(numeric_cols):
    m = hide[:, j]
    if m.sum() == 0:
        continue
    scale = scaler.scale_[j]  # back to original units
    truth = sample[m, j] * scale
    rows.append({
        "feature": col,
        "hidden_values": int(m.sum()),
        "KNN_MAE": np.abs(knn_filled[m, j] * scale - truth).mean(),
        "KNN_RMSE": np.sqrt(((knn_filled[m, j] * scale - truth) ** 2).mean()),
        "MeanFill_MAE": np.abs(mean_filled[m, j] * scale - truth).mean(),
        "MeanFill_RMSE": np.sqrt(((mean_filled[m, j] * scale - truth) ** 2).mean()),
    })
validation = pd.DataFrame(rows).set_index("feature").round(3)
validation["RMSE_improvement_%"] = (100 * (1 - validation["KNN_RMSE"] / validation["MeanFill_RMSE"])).round(1)
display(validation)
validation.to_csv(RESULTS_DIR / "imputation_validation.csv")
summary["knn_vs_mean_median_rmse_improvement_pct"] = float(validation["RMSE_improvement_%"].median())

ax = validation[["KNN_RMSE", "MeanFill_RMSE"]].div(validation["MeanFill_RMSE"], axis=0).plot.bar(figsize=(12, 5))
ax.set_ylabel("RMSE relative to mean-fill (lower is better)")
ax.set_title("Imputation hold-out test: KNN vs. mean fill")
plt.tight_layout()
save_fig("07_imputation_validation")
plt.show()

## 10. Categorical imputation and encoding

* **Wind directions:** filled forward then backward **within each station** (rows are sorted by date), so one station never
  borrows another station's wind. Any leftover gaps use the global mode.
  Compass points are circular (N is next to NNW), so they are encoded as `sin`/`cos` of the bearing instead of arbitrary integers.
* **`RainToday`:** by the dataset's definition it is "Yes" when rainfall is at least 1 mm, so missing values are derived from the imputed `Rainfall`.
* **`RainTomorrow`:** mapped to 0/1 (no missing values remain after Section 7).
* **`Location`:** label-encoded (mapping saved). Use one-hot encoding for linear models.

In [ ]:
cat_df = weather_df_clean[categorical_cols].copy()

COMPASS = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
wind_features = {}
for col in wind_cols:
    filled = cat_df.groupby(weather_df_clean["Location"])[col].ffill()
    filled = filled.groupby(weather_df_clean["Location"]).bfill()
    filled = filled.fillna(filled.mode()[0])
    angle = filled.map({d: i * 2 * np.pi / 16 for i, d in enumerate(COMPASS)})
    wind_features[f"{col}_sin"] = np.sin(angle)
    wind_features[f"{col}_cos"] = np.cos(angle)
wind_features = pd.DataFrame(wind_features, index=cat_df.index)

rain_today = cat_df["RainToday"].map({"No": 0, "Yes": 1})
rain_today = rain_today.fillna((numeric_filled["Rainfall"] >= 1.0).astype(int)).astype(int)
rain_tomorrow = cat_df["RainTomorrow"].map({"No": 0, "Yes": 1}).astype(int)

le = LabelEncoder()
location_code = pd.Series(le.fit_transform(cat_df["Location"]), index=cat_df.index, name="Location_code")
location_map = {name: int(code) for name, code in zip(le.classes_, le.transform(le.classes_))}
(RESULTS_DIR / "location_encoding.json").write_text(json.dumps(location_map, indent=2))

## 11. Assemble, validate and export

In [ ]:
weather_df_cleaned = pd.concat(
    [weather_df_clean[["Date", "Location"]], location_code, numeric_filled, wind_features,
     rain_today.rename("RainToday"), rain_tomorrow.rename("RainTomorrow")],
    axis=1,
)

assert weather_df_cleaned.isna().sum().sum() == 0, "NaNs remain in the final table"
assert len(weather_df_cleaned) == len(weather_df_clean)

print("Final shape:", weather_df_cleaned.shape)
balance = weather_df_cleaned["RainTomorrow"].value_counts(normalize=True).mul(100).round(2)
print("\nRainTomorrow class balance (%):")
print(balance.to_string())

summary.update(
    final_rows=int(weather_df_cleaned.shape[0]),
    final_columns=int(weather_df_cleaned.shape[1]),
    rain_tomorrow_yes_pct=float(balance.get(1, 0)),
    rain_tomorrow_no_pct=float(balance.get(0, 0)),
    knn={"n_neighbors": KNN_NEIGHBORS, "weights": "distance", "metric": "nan_euclidean"},
)
weather_df_cleaned.head(10)

In [ ]:
# Class balance and missing-data rows are plotted so the model-ready table is easy to review
ax = balance.rename({0: "No rain", 1: "Rain"}).plot.bar(figsize=(5, 4), rot=0, color=["#4c78a8", "#f58518"])
ax.set_ylabel("% of days")
ax.set_title("RainTomorrow class balance (cleaned data)")
plt.tight_layout()
save_fig("08_class_balance")
plt.show()

weather_df_cleaned.to_csv(OUT_DIR / "weatherAUS_cleaned.csv", index=False)
(RESULTS_DIR / "preprocessing_summary.json").write_text(json.dumps(summary, indent=2))
print("Saved cleaned data and results/preprocessing_summary.json")

## Next steps

The table above is model-ready. Suggested next step: a time-ordered train/test split (train on earlier years, test on the latest),
a `Pipeline` that fits the scaler/imputer on the training split only, and a baseline classifier compared against a
"predict no rain" benchmark using **recall/F1/ROC-AUC** rather than accuracy, since the classes are imbalanced.